# XXI data check

Work through four questions: pathology without neck dissection, missing
observations, consistency, and the meaning of the recorded consensus.

The calculations are shared in `scripts/data_checks.py`; this notebook selects
the XXI workbook and presents its results. **Run all cells** to refresh the report.
All patients remain included. Expand a code cell, evidence table or settings panel
when you need more detail.

An observation is one **patient–side–level** value. The main-level summaries use
**I, II, III, IV, V and VII on both sides**. Unknown values remain unknown.

In [1]:
DATASET_NAME = "XXI"
WORKBOOK_PATH = "updated_datasets/XXI_update.xlsx"

In [2]:
from pathlib import Path
import sys

import pandas as pd
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "updated_datasets").exists():
    PROJECT_DIR = PROJECT_DIR / "Mexico_data_analysis"

if str(PROJECT_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR.resolve()))

import importlib
from scripts import data_checks as checks
importlib.reload(checks)
from scripts import notebook_display as report
importlib.reload(report)
from scripts.notebook_display import SOURCE_LABELS, comparison_view, show_note, show_table

DATA_FILE = PROJECT_DIR / WORKBOOK_PATH
source_data = pd.read_excel(DATA_FILE, header=[0, 1, 2])
analysis = checks.analyse_dataset(source_data)
raw = analysis["raw"]
without = "max_llh_without_pathology"
with_pathology = "max_llh_with_pathology"
show_note(f"{DATASET_NAME} · {len(raw)} patients", f"Source: {DATA_FILE.name}")

## 1. Pathology without neck dissection

Find patients with at least one recorded pathology result whose neck-dissection
flag is False or missing. Both positive and negative pathology count, including
results recorded only at sublevels.

Read each patient's recorded levels alongside the centre's comments. A biopsy
can explain pathology without a neck dissection; the finding is a request for
context, not an automatic correction of the flag.

In [3]:
pathology_without_neck_dissection = analysis["pathology_without_neck_dissection"]
show_note(
    f"{len(pathology_without_neck_dissection)} of {len(raw)} patients have pathology without dissection marked True",
    "This can reflect a biopsy. Review the centre's comments before changing a flag.",
)

def compact_pathology(text):
    by_side = {}
    for item in text.split("; "):
        side, observation = item.split(" ", 1)
        level, value = observation.split("=", 1)
        by_side.setdefault(side, []).append(level + (" +" if value == "positive" else " −"))
    return "; ".join(f"{side}: {', '.join(values)}" for side, values in by_side.items())

pathology_view = pathology_without_neck_dissection.copy()
pathology_view["neck_dissection"] = pathology_view["neck_dissection"].map(
    {True: "Yes", False: "No"}
).fillna("Unknown")
pathology_view["pathology_observations"] = pathology_view["pathology_observations"].map(compact_pathology)
show_table(pathology_view.rename(columns={
    "patient_id": "Patient", "neck_dissection": "Neck dissection",
    "pathology_observations": "Pathology (+ / −)", "comments": "Centre's comments",
}))

Patient,Neck dissection,Pathology (+ / −),Centre's comments
XX I-1,No,"ipsi: I +, Ia +, Ib +, II +, IIa +, IIb +, III +, IV −, V −, VII −; contra: I +, Ia +, Ib +, II +, IIa +, IIb +, III +, IV −, V −, VII −","Here you can write additional information if needed. If you want, you can also add additional columns next to a diagnosis for example to make comments on the diagnosis"
XX I-2,No,"ipsi: I +, Ia +, Ib +, II +, IIa +, IIb +, III +, IV +, V −, VII −; contra: I +, Ia +, Ib +, II +, IIa +, IIb +, III +, IV +, V −, VII −",—
XX I-3,No,"ipsi: I +, Ia +, Ib +, II +, IIa +, IIb +, III +, IV +, V −, VII −; contra: I +, Ia +, Ib +, II +, IIa +, IIb +, III +, IV +, V −, VII −",—
XX I-4,No,"ipsi: I +, Ia +, Ib +, II +, IIa +, IIb +, III +, IV −, V −, VII −; contra: I +, Ia +, Ib +, II +, IIa +, IIb +, III +, IV −, V −, VII −",—
XX I-5,No,"ipsi: I −, Ia −, Ib −, II +, IIa +, IIb +, III +, IV +, V +, VII −; contra: I −, Ia −, Ib −, II +, IIa +, IIb +, III +, IV +, V +, VII −",—
XX I-6,No,"ipsi: I −, Ia −, Ib −, II +, IIa +, IIb +, III +, IV +, V −, VII −; contra: I −, Ia −, Ib −, II +, IIa +, IIb +, III +, IV +, V −, VII −",—
XX I-8,No,"ipsi: I +, Ia +, Ib +, II +, IIa +, IIb +, III +, IV −, V −, VII −; contra: I +, Ia +, Ib +, II +, IIa +, IIb +, III +, IV −, V −, VII −",—
XX I-9,No,"ipsi: I +, Ia +, Ib +, II +, IIa +, IIb +, III +, IV −, V −, VII −; contra: I +, Ia +, Ib +, II +, IIa +, IIb +, III +, IV −, V −, VII −",—
XX I-10,No,"ipsi: I +, Ia +, Ib +, II +, IIa +, IIb +, III +, IV +, V −, VII −; contra: I +, Ia +, Ib +, II +, IIa +, IIb +, III +, IV +, V −, VII −",—
XX I-11,No,"ipsi: I +, Ia +, Ib +, II +, IIa +, IIb +, III +, IV +, V +, VII −; contra: I +, Ia +, Ib +, II +, IIa +, IIb +, III +, IV −, V −, VII −",—


## 2. Missing observations

For each source, count patients with any data and known versus missing main-level
values. There are **12 possible values per patient**: six levels on each side.
An absent modality block is marked separately from a block with missing entries.

Coverage describes the information available for later comparisons. It does not
imply that every patient should have undergone every test. Missing observations
are never counted as negative results.

In [4]:
missing_observations = analysis["missing_observations"]
missing_view = missing_observations.copy()
missing_view["Patients with data"] = missing_view["patients_with_data"].map(lambda n: f"{n}/{len(raw)}")
missing_view["Missing values"] = missing_view["missing_pairs"].map(
    lambda n: f"{n}/{len(raw) * 12}"
)
missing_view["modality"] = missing_view["modality"].replace({"diagnostic_consensus": "Recorded consensus"})
missing_view.loc[missing_view["columns_present"].eq(0), "modality"] += " (absent)"
show_table(missing_view[["modality", "Patients with data", "Missing values", "coverage_pct"]].rename(
    columns={"modality": "Source", "coverage_pct": "Coverage"}
), percentages=["Coverage"])

Source,Patients with data,Missing values,Coverage
Recorded consensus,17/17,0/204,100.0%
pathology,17/17,0/204,100.0%
CT,17/17,0/204,100.0%
MRI,1/17,192/204,5.9%
PET,2/17,180/204,11.8%
FNA,17/17,0/204,100.0%
pCT (absent),0/17,204/204,0.0%


## 3. Consistency

Check I against Ia/Ib and II against IIa/IIb within each modality. A positive
child requires a positive parent; two negative children require a negative
parent. One negative child with an unknown sibling is insufficient.

Also run the same reduced checks for identity, valid values, N-stage and node
counts used in the original notebook. Findings are grouped below. Invalid binary
entries are reported here and treated as unknown in comparisons and reconstruction.
Consensus–pathology disagreements are examined separately in section 4.

In [5]:
hierarchy_issues = analysis["hierarchy_issues"]
sanity_issues = analysis["sanity_issues"]
consistency_summary = analysis["consistency_summary"]

total_flags = len(hierarchy_issues) + len(sanity_issues)
show_note(
    "No consistency flags" if total_flags == 0 else f"{total_flags} consistency findings",
    f"Main/sublevel conflicts: {len(hierarchy_issues)}. Reduced sanity findings: {len(sanity_issues)}.",
)
if not hierarchy_issues.empty:
    show_table(hierarchy_issues, details="Main/sublevel findings")
if not sanity_issues.empty:
    show_table(sanity_issues[["patient_id", "check", "category", "details"]],
               details="Reduced sanity-check findings")

## 4. What does the recorded consensus represent?

### Recorded consensus versus each source

Compare the recorded consensus with pathology, CT, MRI, PET, FNA and pCT at the
same patient, side and main level. Only pairs with **both values known** enter
the denominator; the patient column counts people with a disagreement.

Agreement percentages include both positive and negative agreement. High overall
agreement can coexist with important positive-case disagreements, so the case
evidence below remains essential. No comparable pairs means no percentage.

In [6]:
recorded_vs_modalities = analysis["recorded_vs_modalities"]
pathology_result = recorded_vs_modalities.iloc[0]
show_note(
    f"Consensus–pathology disagreements: {int(pathology_result['disagreements'])} / "
    f"{int(pathology_result['compared_pairs'])} known pairs",
    f"Patients affected: {int(pathology_result['patients_affected'])}. Individual cases appear below.",
)
show_table(comparison_view(recorded_vs_modalities, ["Pathology", "CT", "MRI", "PET", "FNA", "pCT"]),
           percentages=["Agreement"])

Comparison,Known pairs,Disagreements,Patients affected,Agreement
Pathology,204,0,0,100.0%
CT,204,19,7,90.7%
MRI,12,2,1,83.3%
PET,24,0,0,100.0%
FNA,204,6,3,97.1%
pCT,0,0,0,—


### Reconstruct consensus with and without pathology

**Without pathology:** CT, MRI, PET, pCT and FNA, where available. **With pathology:**
the same inputs plus pathology. Both use `max_llh`; the recorded consensus is
excluded from both, and FNA remains included.


<details>
<summary>Reconstruction method notes</summary>

Both runs use the same installed lydata defaults and I/II subdivisions. The source observations are preserved. Parent or child labels can be inferred; otherwise, no input means an unknown label.

Default pathology sensitivity and specificity are 1.0. Agreement of a reconstruction containing pathology with that same input is expected by construction.

</details>

In [7]:
modality_settings = analysis["modality_settings"]
reconstruction_summary = analysis["reconstruction_summary"]
show_table(comparison_view(reconstruction_summary, [
    "Recorded vs without pathology", "Recorded vs with pathology", "Without vs with pathology",
]), percentages=["Agreement"])

settings_view = modality_settings.rename(columns={
    "modality": "Modality", "available": "Present", "sensitivity": "Sensitivity",
    "specificity": "Specificity",
})
settings_view["Present"] = settings_view["Present"].map({True: "Yes", False: "No"})
show_table(settings_view, details="Model assumptions: sensitivity and specificity")

Comparison,Known pairs,Disagreements,Patients affected,Agreement
Recorded vs without pathology,204,6,3,97.1%
Recorded vs with pathology,204,0,0,100.0%
Without vs with pathology,204,6,3,97.1%


Modality,Present,Sensitivity,Specificity
CT,Yes,0.81,0.76
MRI,Yes,0.81,0.63
PET,Yes,0.79,0.86
pCT,No,0.81,0.86
FNA,Yes,0.8,0.98
pathology,Yes,1,1


### Which reconstruction agrees with the recorded consensus?

First identify known labels that change when pathology is added. Among those
with known pathology and recorded consensus, count how often the recorded
consensus agrees with each reconstruction. This focuses on the observations
that can distinguish a clinical consensus from one incorporating pathology.

The main case table includes disagreements involving pathology or the
reconstructed consensuses. Ordinary differences with an individual modality
appear only in the expandable full evidence table.
The expanded pathology comparison uses the same known pairs for all three
consensuses. Agreement alone cannot confirm the intended definition or timing.

**+** positive · **−** negative · **—** unknown.

The **Hierarchy** column explains parent changes, for example
`I − → + (Ia −, Ib +)`. Expand **Parent and sublevel evidence: before
and after hierarchy** below the main table to inspect the raw sublevels
and each reconstruction before and after reconciliation.


In [8]:
matched_pathology_summary = analysis["matched_pathology_summary"]
reconstruction_changed = analysis["reconstruction_changed"]
interpretation_summary = analysis["interpretation_summary"]
show_note(
    f"Labels changed by pathology: {int(reconstruction_changed.to_numpy().sum())}",
    f"Patients affected: {int(reconstruction_changed.any(axis=1).sum())}.",
)
interpretation_view = pd.DataFrame({
    "Reconstruction": ["Without pathology", "With pathology"],
    "Agrees with recorded consensus on changed pairs": [
        f"{row.compared_pairs - row.disagreements}/{row.compared_pairs}"
        if row.compared_pairs else "—"
        for row in interpretation_summary.itertuples(index=False)
    ],
})
show_table(interpretation_view)

# Evidence rendering is shared across all dataset notebooks.
report.show_disagreement_evidence(analysis)
show_table(comparison_view(matched_pathology_summary, [
    "Recorded consensus vs pathology", "Without pathology vs pathology", "With pathology vs pathology",
]), percentages=["Agreement"], details="Agreement with pathology on the same known pairs")


Reconstruction,Agrees with recorded consensus on changed pairs
Without pathology,0/6
With pathology,6/6


Patient,Location,Hierarchy,Recorded,Pathology,Without path.,With path.,Raw modalities
XX I-8,ipsi III,—,+,+,−,+,CT III −; FNA III −
XX I-8,contra III,—,+,+,−,+,CT III −; FNA III −
XX I-9,contra II,—,+,+,−,+,"CT II −, IIa −, IIb −; FNA II −, IIa −, IIb −"
XX I-9,contra III,—,+,+,−,+,CT III −; FNA III −
XX I-14,ipsi IV,—,+,+,−,+,CT IV −; MRI IV −; FNA IV −
XX I-14,contra IV,—,+,+,−,+,CT IV −; MRI IV −; FNA IV −


Patient,Side,Level,Recorded consensus,Pathology,CT,MRI,PET,FNA,Without pathology,With pathology,Centre's comments
XX I-1,ipsi,III,+,+,−,<NA>,<NA>,+,+,+,"Here you can write additional information if needed. If you want, you can also add additional columns next to a diagnosis for example to make comments on the diagnosis"
XX I-1,contra,III,+,+,−,<NA>,<NA>,+,+,+,"Here you can write additional information if needed. If you want, you can also add additional columns next to a diagnosis for example to make comments on the diagnosis"
XX I-8,ipsi,I,+,+,−,<NA>,<NA>,+,+,+,—
XX I-8,ipsi,III,+,+,−,<NA>,<NA>,−,−,+,—
XX I-8,contra,I,+,+,−,<NA>,<NA>,+,+,+,—
XX I-8,contra,III,+,+,−,<NA>,<NA>,−,−,+,—
XX I-9,ipsi,I,+,+,−,<NA>,<NA>,+,+,+,—
XX I-9,contra,I,+,+,−,<NA>,<NA>,+,+,+,—
XX I-9,contra,II,+,+,−,<NA>,<NA>,−,−,+,—
XX I-9,contra,III,+,+,−,<NA>,<NA>,−,−,+,—


Patient,Location,Source,Stage,Parent,Sublevels
XX I-8,ipsi I,Recorded consensus,Recorded,+,"Ia +, Ib +"
XX I-8,ipsi I,Pathology,Recorded,+,"Ia +, Ib +"
XX I-8,ipsi I,CT,Recorded,−,"Ia −, Ib −"
XX I-8,ipsi I,FNA,Recorded,+,"Ia +, Ib +"
XX I-8,ipsi I,Without pathology,Before hierarchy,+,"Ia +, Ib +"
XX I-8,ipsi I,Without pathology,After hierarchy,+,"Ia +, Ib +"
XX I-8,ipsi I,With pathology,Before hierarchy,+,"Ia +, Ib +"
XX I-8,ipsi I,With pathology,After hierarchy,+,"Ia +, Ib +"
XX I-8,contra I,Recorded consensus,Recorded,+,"Ia +, Ib +"
XX I-8,contra I,Pathology,Recorded,+,"Ia +, Ib +"


Comparison,Known pairs,Disagreements,Patients affected,Agreement
Recorded consensus vs pathology,204,0,0,100.0%
Without pathology vs pathology,204,6,3,97.1%
With pathology vs pathology,204,0,0,100.0%


### N-stage versus involvement

Compare the recorded N-stage with involvement from the recorded consensus,
both reconstructions and pathology. Flag **N0 with a positive level** and
**N+ without an observed positive**. Distinguish complete negative coverage
from partial negative or absent observations.

The expanded overview lists positive levels on each side for every patient.
The N-stage basis remains unconfirmed unless its own field or prefix specifies
it; a T-stage prefix does not establish an N-stage basis. These are N0/N+
review prompts, not a reconstruction of exact TNM substages. Different sources
may also describe different assessments.

In [9]:
n_stage_details = analysis["n_stage_details"]
n_stage_overview = analysis["n_stage_overview"]
n_stage_review = analysis["n_stage_review"]
review_patients = n_stage_review["patient_id"].unique()
show_note(
    f"N-stage review: {len(review_patients)} " + ("patient" if len(review_patients) == 1 else "patients"),
    ", ".join(map(str, review_patients)) if len(review_patients) else "No N0/N+ review findings.",
)
if not n_stage_review.empty:
    stage_review_view = n_stage_review[[
        "patient_id", "n_stage", "source", "known_pairs", "assessment"
    ]].copy()
    stage_review_view["source"] = stage_review_view["source"].replace(SOURCE_LABELS)
    stage_review_view["known_pairs"] = stage_review_view["known_pairs"].map(lambda n: f"{n}/12")
    stage_review_view["assessment"] = stage_review_view["assessment"].str.removeprefix("Review: ")
    show_table(stage_review_view.rename(columns={
        "patient_id": "Patient", "n_stage": "N-stage", "source": "Source",
        "known_pairs": "Observed", "assessment": "Review finding",
    }))
show_table(n_stage_overview.rename(columns={
    "patient_id": "Patient", "n_stage": "N-stage", "stage_basis": "N-stage basis", **SOURCE_LABELS,
}), details="All patients: N-stage and involved levels on each side")

Patient,N-stage,N-stage basis,Recorded consensus,Without pathology,With pathology,Pathology
XX I-1,1,unconfirmed,"ipsi: I, II, III; contra: I, II, III","ipsi: I, II, III; contra: I, II, III","ipsi: I, II, III; contra: I, II, III","ipsi: I, II, III; contra: I, II, III"
XX I-2,2c,unconfirmed,"ipsi: I, II, III, IV; contra: I, II, III, IV","ipsi: I, II, III, IV; contra: I, II, III, IV","ipsi: I, II, III, IV; contra: I, II, III, IV","ipsi: I, II, III, IV; contra: I, II, III, IV"
XX I-3,2c,unconfirmed,"ipsi: I, II, III, IV; contra: I, II, III, IV","ipsi: I, II, III, IV; contra: I, II, III, IV","ipsi: I, II, III, IV; contra: I, II, III, IV","ipsi: I, II, III, IV; contra: I, II, III, IV"
XX I-4,2c,unconfirmed,"ipsi: I, II, III; contra: I, II, III","ipsi: I, II, III; contra: I, II, III","ipsi: I, II, III; contra: I, II, III","ipsi: I, II, III; contra: I, II, III"
XX I-5,2c,unconfirmed,"ipsi: II, III, IV, V; contra: II, III, IV, V","ipsi: II, III, IV, V; contra: II, III, IV, V","ipsi: II, III, IV, V; contra: II, III, IV, V","ipsi: II, III, IV, V; contra: II, III, IV, V"
XX I-6,2a,unconfirmed,"ipsi: II, III, IV; contra: II, III, IV","ipsi: II, III, IV; contra: II, III, IV","ipsi: II, III, IV; contra: II, III, IV","ipsi: II, III, IV; contra: II, III, IV"
XX I-7,2,unconfirmed,"ipsi: I, II, III; contra: none observed","ipsi: I, II, III; contra: none observed","ipsi: I, II, III; contra: none observed","ipsi: I, II, III; contra: none observed"
XX I-8,2b,unconfirmed,"ipsi: I, II, III; contra: I, II, III","ipsi: I, II; contra: I, II","ipsi: I, II, III; contra: I, II, III","ipsi: I, II, III; contra: I, II, III"
XX I-9,2b,unconfirmed,"ipsi: I, II, III; contra: I, II, III","ipsi: I, II, III; contra: I","ipsi: I, II, III; contra: I, II, III","ipsi: I, II, III; contra: I, II, III"
XX I-10,2c,unconfirmed,"ipsi: I, II, III, IV; contra: I, II, III, IV","ipsi: I, II, III, IV; contra: I, II, III, IV","ipsi: I, II, III, IV; contra: I, II, III, IV","ipsi: I, II, III, IV; contra: I, II, III, IV"


## Workable data

In [10]:
disagreement_report = analysis["disagreements"].copy()
disagreement_report['patient_id'].unique()

<StringArray>
['XX I-8', 'XX I-9', 'XX I-14']
Length: 3, dtype: str

if pathology is okay, this is not an issue! can be ignored after feedback hopefully

In [11]:
data = analysis["raw"]
excluded_ids = disagreement_report["patient_id"].unique()

evaluation_data = data.loc[~data[("patient", "core", "id")].isin(excluded_ids)].copy()

print(f"Retained {len(evaluation_data)} of {len(data)} patients")

Retained 14 of 17 patients


In [12]:
evaluation_data.to_csv(PROJECT_DIR / "ready_tables" / f"{DATASET_NAME}_evaluation_data.csv", index=False)